In [ ]:
import itertools
from pathlib import Path
import re

from matplotlib.colors import CenteredNorm
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pickle
from scipy.stats import ttest_ind
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import make_pipeline

from src.data import get_electrode_df
from src.stimuli import POD_dict

In [ ]:
window_size = 0.1
stride = 0.05
pca_num_components = 0.95
outdir = "."

roi_filters = {
    "stg": ["superiortemporal"],
    "mtg": ["middletemporal"],
    "precentral": ["precentral"],
    "postcentral": ["postcentral"],
    "smg": ["supramarginal"],
}

In [ ]:
all_epoch_paths = list(Path("epochs").glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epochs", str(path))[0]
    if subject_name == "EC282":
        # missing ecog data
        continue
    epochs[subject_name] = mne.read_epochs(str(path)).pick("ecog").resample(100)

In [ ]:
electrode_df = pd.concat([get_electrode_df(subject_name) for subject_name in epochs.keys()],
                         keys=epochs.keys(), names=["subject"])
electrode_df["roi"] = electrode_df.roi.astype(str)
electrode_df = electrode_df.droplevel("electrode_name").set_index("roi", append=True).reorder_levels(["subject", "roi", "electrode_idx"])

In [ ]:
def run_decoding_analysis(epochs, stride, window_size, roi_filter=None):
    global_tmin = 0. # min([epoch.times.min() for epoch in epochs.values()])
    global_tmax = max([epoch.times.max() for epoch in epochs.values()])
    windows_left = np.arange(global_tmin, global_tmax, stride)
    windows_right = np.minimum(global_tmax, windows_left + window_size)
    windows = list(zip(windows_left, windows_right))

    # `outcomes` stores prediction outcomes for each epoch under the optimal model
    outcomes = {}
    # `test_scores` stores cross-validated estimates of held-out generalization
    test_scores = {}
    phoneme_pairs = next(iter(epochs.values())).metadata.phoneme_pair.unique()

    i = 0
    for phoneme_pair, (tmin, tmax), subject_name in tqdm(list(itertools.product(phoneme_pairs, windows, epochs))):
        electrodes_i = electrode_df.loc[subject_name]
        if roi_filter is not None:
            try:
                electrodes_i = electrodes_i.loc[roi_filter]
            except KeyError:
                continue
        electrodes_i = electrodes_i.index.get_level_values("electrode_idx")
        if len(electrodes_i) == 0:
            continue

        epochs_i = epochs[subject_name][f"phoneme_pair == '{phoneme_pair}'"].copy()
        pick_electrodes = list(set(electrodes_i) & set(range(len(epochs_i.ch_names))))

        epochs_i = epochs_i.pick(pick_electrodes).crop(tmin, tmax)
        if len(epochs_i) == 0:
            continue

        # epochs * channels * time
        X = epochs_i.get_data()
        X = X.reshape(X.shape[0], -1)

        y = epochs_i.metadata.word_end.str[0] == epochs_i.metadata.phoneme_pair.str[0]
        step = epochs_i.metadata.resampled

        cv_inner = StratifiedKFold(3, shuffle=True)
        cv_outer = StratifiedKFold(3, shuffle=True)

        pipeline = [StandardScaler()]
        if pca_num_components is not None:
            pipeline.append(PCA(n_components=pca_num_components))
        pipeline.append(LogisticRegressionCV(Cs=10, cv=cv_inner, max_iter=1000))
        model = make_pipeline(*pipeline)
        fitted = cross_validate(model, X, y, cv=cv_outer, scoring="roc_auc", return_estimator=True)

        test_scores[subject_name, phoneme_pair, tmin, tmax] = fitted["test_score"]

        outcomes[subject_name, phoneme_pair, tmin, tmax] = pd.concat({
            fold: pd.DataFrame({"decoder_target": y,
                                "decoder_prediction": estimator.predict(X),
                                "decoder_proba": estimator.predict_proba(X)[:, 1]},
                            index=epochs_i.metadata.index.rename("epoch_idx"))
            for fold, estimator in enumerate(fitted["estimator"])
        }, names=["fold"], keys=range(len(fitted["estimator"])))

    return test_scores, outcomes

In [ ]:
decoding_kwargs = dict(stride=stride, window_size=window_size)
scores_all_elecs, outcomes_all_elecs = run_decoding_analysis(epochs, **decoding_kwargs)

all_scores = {"*": scores_all_elecs}
all_outcomes = {"*": outcomes_all_elecs}

In [ ]:
for name, rois in tqdm(roi_filters.items(), unit="ROI"):
    print(name)
    all_scores[name], all_outcomes[name] = run_decoding_analysis(epochs, roi_filter=rois, **decoding_kwargs)

In [ ]:
scores_df = pd.concat(
    {roi_filter: pd.concat(
        {key: pd.Series(scores_i).rename("roc_auc") for key, scores_i in scores.items()},
        names=["subject", "phoneme_pair", "tmin", "tmax", "fold"])
     for roi_filter, scores in all_scores.items()},
    names=["roi_filter"])

In [ ]:
scores_df.to_csv(Path(outdir) / "scores.csv")

In [ ]:
with (Path(outdir) / "outcomes.pkl").open("wb") as f:
    pickle.dump(all_outcomes, f)

In [ ]:
plot_df = scores_df.reset_index()
plot_df["t_center"] = plot_df.tmin + (plot_df.tmax - plot_df.tmin) / 2

In [ ]:
def plot(by_subject=True):
    col_order = sorted(plot_df.phoneme_pair.unique())
    row_order = sorted(plot_df.roi_filter.unique())
    hue_order = sorted(plot_df.subject.unique())
    kwargs = dict(hue="subject", hue_order=hue_order) if by_subject else {}
    g = sns.relplot(data=plot_df, x="t_center", y="roc_auc",
                    row="roi_filter", row_order=row_order,
                    col="phoneme_pair", col_order=col_order,
                    kind="line", errorbar="se", aspect=2, **kwargs)
    for ax in g.axes.flat:
        ax.axhline(0.5, ls="--", color="gray")
        ax.set_xlim((0, plot_df.tmax.max()))
        ax.set_ylim(0.4, 1.0)
        ax.set_ylabel("ROC AUC")
        ax.set_xlabel("Time from word onset (s)")
    for (row, col, hue), facet_data in g.facet_data():
        phoneme_pair = col_order[col]
        ax = g.axes[row, col]
        ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
        ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
        ax.axvline(POD_dict[phoneme_pair], color="black", linestyle="dotted")
    return g

In [ ]:
g = plot(by_subject=False)
g.savefig(Path(outdir) / "decoding.pdf")

In [ ]:
g = plot(by_subject=True)
g.savefig(Path(outdir) / "decoding_by_subject.pdf")

## Analyze outputs by acoustic step

In [ ]:
odf = pd.concat({roi_filter: pd.concat(all_outcomes_i, names=["subject", "phoneme_pair", "tmin", "tmax"])
                 for roi_filter, all_outcomes_i in all_outcomes.items()},
                names=["roi_filter"])
odf

In [ ]:
all_metadata = pd.concat([epochs_i.metadata.rename_axis("epoch_idx") for epochs_i in epochs.values()],
                         names=["subject"], keys=epochs.keys())
all_metadata

In [ ]:
# decoder proba time series
proba_df = pd.merge(odf, all_metadata[["resampled"]], left_index=True, right_index=True, how="left", validate="1:1").reset_index()
col_order = sorted(proba_df.phoneme_pair.unique())
row_order = sorted(proba_df.roi_filter.unique())
g = sns.relplot(data=proba_df, kind="line",
                col="phoneme_pair", col_order=col_order,
                row="roi_filter", row_order=row_order,
                x="tmin", y="decoder_proba", hue="resampled", errorbar="se")

for ax in g.axes.flat:
    ax.axhline(0.5, ls="--", color="gray")
    ax.set_xlim((0, proba_df.tmax.max()))
    ax.set_ylabel("proba")
    ax.set_xlabel("Time from word onset (s)")
for (row, col, hue), facet_data in g.facet_data():
    phoneme_pair = col_order[col]
    ax = g.axes[row, col]
    ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
    ax.axvline(POD_dict[phoneme_pair], color="black", linestyle="dotted")

g.savefig(Path(outdir) / "proba_by_step.pdf")

In [ ]:
def eval_roc_auc(rows):
    y_true = rows.decoder_target
    y_pred = rows.decoder_proba
    return roc_auc_score(y_true, y_pred)
all_metadata = pd.concat([epochs_i.metadata.rename_axis("epoch_idx") for epochs_i in epochs.values()],
                         names=["subject"], keys=epochs.keys())
roc_auc_df = pd.merge(odf, all_metadata[["resampled"]], left_index=True, right_index=True, how="left", validate="1:1") \
    .groupby(["subject", "roi_filter", "phoneme_pair", "tmin", "tmax", "resampled", "fold"]) \
    .apply(eval_roc_auc).rename("roc_auc").reset_index()

In [ ]:
col_order = sorted(roc_auc_df.phoneme_pair.unique())
row_order = sorted(roc_auc_df.roi_filter.unique())
g = sns.relplot(data=roc_auc_df, kind="line",
                col="phoneme_pair", col_order=col_order,
                row="roi_filter", row_order=row_order,
                x="tmin", y="roc_auc",
                hue="resampled", palette="coolwarm", hue_norm=CenteredNorm(vcenter=3.5),
                errorbar="se")

for ax in g.axes.flat:
    ax.axhline(0.5, ls="--", color="gray")
    ax.set_xlim((0, roc_auc_df.tmax.max()))
    ax.set_ylabel("ROC AUC")
    ax.set_xlabel("Time from word onset (s)")
for (row, col, hue), facet_data in g.facet_data():
    phoneme_pair = col_order[col]
    ax = g.axes[row, col]
    ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
    ax.axvline(POD_dict[phoneme_pair], color="black", linestyle="dotted")

g.savefig(Path(outdir) / "decoding_by_step.pdf")